# LoRA Hyperparameter Sensitivity Analysis – Google Colab

Identifies which LoRA hyperparameters (`lora_r`, `lora_alpha`, `lora_dropout`, `learning_rate`, `use_ffn`)
have the largest impact on fine-tuning performance of the pretrained **NeoQuasar/Kronos-base** model.

**Workflow:**
1. Setup environment
2. Prepare training data (arrow format)
3. Random search: sample LoRA configs → brief fine-tune → evaluate
4. Random Forest analysis to rank parameter importance
5. Download results

## Setup: Clone Repository

In [ ]:
!rm -rf /content/BA
print("Cleaned up.")

In [ ]:
import os
from pathlib import Path

repo_url = "https://github.com/bp571/BA"

%cd /content
!git clone {repo_url}
%cd /content/BA

# Tiingo API key
TIINGO_API_KEY = "312c6dab6a1fe6258bbc6652bcdec49a14ee08ad"
os.environ["TIINGO_API_KEY"] = TIINGO_API_KEY
Path(".env").write_text(f"TIINGO_API_KEY={TIINGO_API_KEY}\n")
print("✅ API key configured")

# Clone Kronos (gitignored in main repo)
!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("✅ Kronos cloned")

## Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm einops \
    peft transformers huggingface_hub \
    gluonts python-dotenv tiingo

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Prepare Training Data

`.arrow` files are gitignored and must be generated from the raw asset data.
This downloads OHLCV data via Tiingo and converts it to the Kronos format.

In [ ]:
!python 02_finetuning/training/prepare_data_kronos.py
print("\n✅ Training data ready")

In [ ]:
# Verify arrow files exist
from pathlib import Path
for p in ["data/processed/train_data_kronos.arrow", "data/processed/val_data_kronos.arrow"]:
    size = Path(p).stat().st_size / 1024 / 1024 if Path(p).exists() else None
    status = f"{size:.1f} MB" if size else "MISSING"
    print(f"  {p}: {status}")

## Configuration

In [ ]:
N_SAMPLES   = 60    # Number of random LoRA configurations to evaluate
TRAIN_STEPS = 150   # Fine-tuning steps per configuration
CONTEXT     = 120   # Context window (optimal from data_parameters analysis)
FORECAST    = 6     # Forecast horizon (optimal from data_parameters analysis)
SEED        = 42

print(f"Samples:     {N_SAMPLES}")
print(f"Train steps: {TRAIN_STEPS}")
print(f"Context:     {CONTEXT}")
print(f"Forecast:    {FORECAST}")
print(f"Estimated time on A100: ~{N_SAMPLES * 2} min")

## Step 1: LoRA Random Search

For each of the `N_SAMPLES` configurations:
- Sample: `lora_r` ∈ {4, 8, 16, 32}, `lora_alpha` ∈ {8, 16, 32, 64}, `lora_dropout` ∈ {0.0, 0.05, 0.1, 0.2}, `learning_rate` ∈ {5e-5, 1e-4, 3e-4}, `use_ffn` ∈ {0, 1}
- Fine-tune pretrained Kronos for `TRAIN_STEPS` steps
- Evaluate on 2019–2020 validation data (rolling windows)

In [ ]:
!python 03_sensitivity_analysis/lora_parameters/run_lora_sensitivity.py \
    --n-samples {N_SAMPLES} \
    --train-steps {TRAIN_STEPS} \
    --context {CONTEXT} \
    --forecast {FORECAST} \
    --seed {SEED}

## Step 2: Random Forest Analysis

Trains a Random Forest on the results to rank LoRA parameter importance.

In [ ]:
csv_path = f"03_sensitivity_analysis/lora_parameters/results/lora_search_{N_SAMPLES}.csv"

!python 03_sensitivity_analysis/lora_parameters/analyze_lora_rf.py --csv {csv_path}

## Results

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
print(f"Successful samples: {len(df)}\n")
print(df.describe().round(4))
print("\nBest configs by RankIC:")
display(df.sort_values('rankic', ascending=False).head(5))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

results_dir = Path("03_sensitivity_analysis/lora_parameters/results")

for metric in ['mae', 'rankic']:
    img = results_dir / f"rf_importance_lora_{metric}.png"
    if img.exists():
        print(f"\nFeature Importance – {metric.upper()}:")
        display(Image(str(img)))

## Download Results

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

zip_name = "lora_sensitivity_results.zip"
results_dir = Path("03_sensitivity_analysis/lora_parameters/results")

with zipfile.ZipFile(zip_name, 'w') as zf:
    for f in results_dir.iterdir():
        if f.suffix in ('.csv', '.png'):
            zf.write(f, arcname=f.name)

files.download(zip_name)
print(f"Downloaded: {zip_name}")